# 03B. Topography Feature Extraction (Fixed 90m Resolution)

Floods in Chikwawa are heavily influenced by the extreme low-lying terrain and proximity to the Shire River.
In this notebook, we use Google Earth Engine to download the NASA SRTM Digital Elevation Model (DEM) clipped to the Chikwawa boundary, and compute the `Elevation_m` and `Slope_deg` features.

**Resolution is set to 90m to successfully download the file under Earth Engine limits.**

In [1]:
import ee
import geopandas as gpd
import json
import os
import requests
from pathlib import Path

# 1. Initialize Earth Engine
try:
    ee.Initialize()
    print('Earth Engine initialized successfully.')
except Exception as e:
    print('Please run notebook 02 to set up authentication first.')
    raise e


Earth Engine initialized successfully.


In [2]:
# 2. Load Chikwawa boundary and convert to EE geometry
notebook_dir  = Path(os.path.abspath(''))
project_root  = notebook_dir.parent
boundary_path = project_root / 'data' / 'raw' / 'chikwawa_boundary.geojson'
output_path   = project_root / 'data' / 'raw' / 'chikwawa_topography.tif'

print(f'Boundary file  : {boundary_path}')
print(f'Output will be : {output_path}')

if not boundary_path.exists():
    raise FileNotFoundError(f'Boundary not found at {boundary_path}. Run notebook 01 first.')

chikwawa_gdf = gpd.read_file(boundary_path)

# Convert to EE geometry via GeoJSON
geojson = json.loads(chikwawa_gdf.to_json())
aoi = ee.FeatureCollection(geojson).geometry()
print('Area of Interest (AOI) converted to Earth Engine geometry.')


Boundary file  : c:\Users\linga\Music\Chikwawa-Flood-Prediction\data\raw\chikwawa_boundary.geojson
Output will be : c:\Users\linga\Music\Chikwawa-Flood-Prediction\data\raw\chikwawa_topography.tif
Area of Interest (AOI) converted to Earth Engine geometry.


In [3]:
# 3. Fetch Elevation and Slope from NASA SRTM DEM
dataset    = ee.Image('USGS/SRTMGL1_003')
elevation  = dataset.select('elevation')
slope      = ee.Terrain.slope(elevation)
topo_image = elevation.rename('Elevation_m').addBands(slope.rename('Slope_deg')).clip(aoi)

# Pull real min/max stats to confirm connectivity
stats = topo_image.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=aoi,
    scale=300,
    maxPixels=1e9
).getInfo()
print('Topography stats for Chikwawa:')
for k, v in stats.items():
    print(f'  {k}: {v}')


Topography stats for Chikwawa:
  Elevation_m_max: 917
  Elevation_m_min: 48
  Slope_deg_max: 28.719334834646745
  Slope_deg_min: 0


In [4]:
# 4. Download at 90m resolution (bypassing geemap bug and the 50MB limit)
SCALE = 90  # metres per pixel

print(f'Requesting download URL from Earth Engine (scale={SCALE}m)...')
url = topo_image.getDownloadURL({
    'bands':  ['Elevation_m', 'Slope_deg'],
    'region': aoi,
    'scale':  SCALE,
    'format': 'GEO_TIFF',
    'crs':    'EPSG:4326'
})
print('URL generated. Streaming file to disk...')

response = requests.get(url, stream=True)
response.raise_for_status()

total = 0
with open(output_path, 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
        total += len(chunk)

# Verify the file was actually saved
if output_path.exists() and output_path.stat().st_size > 0:
    size_mb = output_path.stat().st_size / (1024 * 1024)
    print()
    print(f'SUCCESS! File saved to: {output_path}')
    print(f'File size : {size_mb:.2f} MB')
    print(f'Resolution: {SCALE}m per pixel')
else:
    print('ERROR: File was not saved. Check the output path.')


Requesting download URL from Earth Engine (scale=90m)...
URL generated. Streaming file to disk...

SUCCESS! File saved to: c:\Users\linga\Music\Chikwawa-Flood-Prediction\data\raw\chikwawa_topography.tif
File size : 3.04 MB
Resolution: 90m per pixel
